In [ ]:
!pip -q install gradio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf

MODEL_PATH = "/content/drive/MyDrive/Brain_Tumor_CNN_Project/models/efficientnetb0_final.keras"

model = tf.keras.models.load_model(MODEL_PATH)

print("Model loaded successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model loaded successfully!


In [ ]:
import tensorflow as tf

def predict_mri(image_path):
    # Load image
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (224, 224))
    image = tf.cast(image, tf.float32)

    # Add batch dimension
    image = tf.expand_dims(image, axis=0)

    # Prediction
    probability = model.predict(image, verbose=0)[0][0]

    if probability >= 0.5:
        prediction = "Tumour"
        confidence = probability * 100
    else:
        prediction = "Non-Tumour"
        confidence = (1 - probability) * 100

    return prediction, confidence

print("Prediction function ready!")

Prediction function ready!


In [ ]:
import gradio as gr

def classify_mri(image_path):
    prediction, confidence = predict_mri(image_path)

    return prediction, f"{confidence:.2f}%"

with gr.Blocks(title="Brain Tumor MRI Classifier") as demo:

    gr.Markdown("# 🧠 Brain Tumor MRI Classifier")
    gr.Markdown("Upload an MRI image and click Predict.")

    image_input = gr.Image(
        type="filepath",
        label="Upload MRI Image"
    )

    predict_button = gr.Button("🔍 Predict")

    prediction_output = gr.Textbox(
        label="Prediction"
    )

    confidence_output = gr.Textbox(
        label="Confidence"
    )

    predict_button.click(
        fn=classify_mri,
        inputs=image_input,
        outputs=[prediction_output, confidence_output]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a5e329e490600104c2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import os

MODEL_PATH = "/content/drive/MyDrive/Brain_Tumor_CNN_Project/models/efficientnetb0_final.keras"

size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)

print(f"Model size: {size_mb:.2f} MB")

Model size: 16.27 MB
